# Assignment #1: Building an inverted index
Author: Pierre Nugues

## Objectives

The objectives of this assignment are to:
* Write a program that collects all the words from a set of documents
* Build an index from the words
* Represent a document using the Tf.Idf values
* Write a short report of 1 to 2 pages on the assignment
* Read a description of an industrial system and answer a question on it
* Discuss the potential of corpora in applications

## Submission

When you have written all the missing code and run all the cells, you will upload your notebook to Canvas. Do not erase the content of the cells as we will possibly check your programs manually.
The submission instructions are at the bottom of the notebook.

## Description of the assignment

### Outline

In this lab, you will build an indexer to index all the words in a corpus. Conceptually, an index consists of rows with one word per row and the list of files and positions, where this word occurs. Such a row is called a _posting list_. You will encode the position of a word by the number of characters from the start of the file.
<pre>
word1: file_name pos1 pos2 pos3... file_name pos1 pos2 ...
word2: file_name pos1 pos2 pos3... file_name pos1 pos2 ...
...
</pre>

#### Imports

Some imports. Add others as needed

In [177]:
import math
import os
import regex as re
import requests
from zipfile import ZipFile
import json
import numpy as np

## Corpus

You will create an index for a corpus of Selma Lagerlöf's works: To gather the corpus, you can alternatively:
1. Download the <a href="https://github.com/pnugues/ilppp/raw/master/programs/corpus/Selma.zip">Selma folder</a> and uncompress it. It contains novels by <a href="https://sv.wikipedia.org/wiki/Selma_Lagerl%C3%B6f">Selma Lagerlöf</a>. The text of these novels was extracted from <a href="https://litteraturbanken.se/forfattare/LagerlofS/titlar">Lagerlöf arkivet</a> at <a href="https://litteraturbanken.se/">Litteraturbanken</a>.
2. Or run this cell that will download the corpus and place it in your folder.

In [3]:
# Parameters for Selma dataset
SELMA_URL = "https://github.com/pnugues/ilppp/raw/master/programs/corpus/Selma.zip"

SELMA_FILES = [
    os.path.join("Selma", fname) 
    for fname in 
    [
        "bannlyst.txt", 
        "gosta.txt", 
        "herrgard.txt", 
        "jerusalem.txt", 
        "kejsaren.txt", 
        "marbacka.txt", 
        "nils.txt", 
        "osynliga.txt", 
        "troll.txt"
    ]
]

def download_and_extract_selma():
    """Downloads and unpacks Selma.zip"""
    
    # Download if not all files exist
    req = requests.get(SELMA_URL, stream=True)
    if req.status_code != 200:
        print("Failed to download file, got status: " + req.status_code)
        req.close()
    else:
        with open("Selma.zip", "wb") as fd:
            written = 0
            for chunk in req.iter_content(chunk_size=65536):
                fd.write(chunk)
                written += len(chunk)
                print("Downloading: %d bytes written to Selma.zip" % written)

        print("Selma.zip donwnloaded.")
        req.close()
        
        selma_zipfile = ZipFile("Selma.zip")
        selma_files_to_extract = [zi for zi in selma_zipfile.filelist if not zi.filename.startswith("__") and zi.filename.endswith(".txt")]
        for zi in selma_files_to_extract:
            selma_zipfile.extract(zi)
            print("Extracted: " + zi.filename)
            
        print("Done!")
        
# If not all path exists (all are true), then download
if not all([os.path.exists(fname) for fname in SELMA_FILES]):
    download_and_extract_selma()
else:
    print("Selma has been downloaded.")
    
SELMA_FILES

Downloading: 65536 bytes written to Selma.zip
Downloading: 131072 bytes written to Selma.zip
Downloading: 196608 bytes written to Selma.zip
Downloading: 262144 bytes written to Selma.zip
Downloading: 327680 bytes written to Selma.zip
Downloading: 393216 bytes written to Selma.zip
Downloading: 458752 bytes written to Selma.zip
Downloading: 524288 bytes written to Selma.zip
Downloading: 589824 bytes written to Selma.zip
Downloading: 655360 bytes written to Selma.zip
Downloading: 720896 bytes written to Selma.zip
Downloading: 786432 bytes written to Selma.zip
Downloading: 851968 bytes written to Selma.zip
Downloading: 917504 bytes written to Selma.zip
Downloading: 983040 bytes written to Selma.zip
Downloading: 1048576 bytes written to Selma.zip
Downloading: 1114112 bytes written to Selma.zip
Downloading: 1179648 bytes written to Selma.zip
Downloading: 1245184 bytes written to Selma.zip
Downloading: 1310720 bytes written to Selma.zip
Downloading: 1376256 bytes written to Selma.zip
Download

['Selma\\bannlyst.txt',
 'Selma\\gosta.txt',
 'Selma\\herrgard.txt',
 'Selma\\jerusalem.txt',
 'Selma\\kejsaren.txt',
 'Selma\\marbacka.txt',
 'Selma\\nils.txt',
 'Selma\\osynliga.txt',
 'Selma\\troll.txt']

### Running the indexer (optional)

In a production context, your final program would take a corpus as input (here the Selma Lagerlöf's novels) and create an index of all the words with their positions. You should be able to run it this way:
<pre>$ python indexer.py folder_name</pre>
In this lab, you will write the index in a Jupyter Notebook. The conversion into a Python program is left as an optional exercise.

## Programming the Indexer

To make programming easier, you will split this exercise into five steps:
1. Index one file;
2. Read the content of a folder
3. Create a master index for all the files
4. Use tfidf to represent the documents (novels)
5. Compare the documents of a collection

You will use dictionaries to represent the postings.

### Indexing one file

#### Description

Write a program that reads one document <tt>file_name.txt</tt> and outputs an index file:
            <tt>file_name.json</tt>:
1. The index file will contain all the unique words in the document,
                where each word is associated with the list of its positions in the document.
2. You will represent this index as a dictionary, where the keys will be the words, and
                the values, the lists of positions
3. As words, you will consider all the strings of letters that you will set in lower case.
                You will not index the rest (i.e. numbers, punctuations, or symbols).
4. To extract the words, use **Unicode regular expressions**. Do not use <tt>\w+</tt>,
                for instance, but the Unicode equivalent.
5. The word positions will correspond to the number of characters from the beginning of the file.
                (The word offset from the beginning)
6. You will use the <tt>finditer()</tt> method to find the positions of the words. This will return you match objects, where you will get the matches and the positions with the <tt>group()</tt> and <tt>start()</tt> methods.
7. You will use JSON to write your dictionary in an file.


Below is an excerpt of the index of the `bannlyst.txt` text for the words <i>gjord</i>, <i>uppklarnande</i>, and <i>stjärnor</i>. The data is stored in a dictionary:

<pre>
{...
'gjord': [8600, 183039, 220445],
'uppklarnande': [8617],
'stjärnor': [8641], ...
}
</pre>
where the word <i>gjord</i> occurs three times in the text at positions 8600, 183039, and 220445, <i>uppklarnande</i>, once at position 8617, and <i>stjärnor</i>, once at position 8641.

#### Writing a tokenizer 

Write a **Unicode** regular expression to find words defined as sequences of letters. Using the Unicode notation is compulsory. If you are not sure, please refer to the lecture.

In [20]:
# Write your regex here
regex = r"[A-ö]+"

In [21]:
re.findall(regex, 'En gång hade de på Mårbacka en barnpiga, som hette Back-Kajsa 23')

['En',
 'gång',
 'hade',
 'de',
 'på',
 'Mårbacka',
 'en',
 'barnpiga',
 'som',
 'hette',
 'Back',
 'Kajsa']

Using `regex`, write `tokenize(text)` function to tokenize a text. Return their positions.

In [51]:
index = re.finditer('ab*c', "hej, abbc, ac")
for i in index :
    print(i.captures()[0])
    print(i.span())

j = [] + [1] + [2]
j += [2]
j += [3 ,4]
j

abbc
(5, 9)
ac
(11, 13)


[1, 2, 2, 3, 4]

In [27]:
# Write your code here
def tokenize(text):
    return re.finditer(regex, text)

In [28]:
tokens = tokenize('En gång hade de på Mårbacka en barnpiga, som hette Back-Kajsa.')
list(tokens)

[<regex.Match object; span=(0, 2), match='En'>,
 <regex.Match object; span=(3, 7), match='gång'>,
 <regex.Match object; span=(8, 12), match='hade'>,
 <regex.Match object; span=(13, 15), match='de'>,
 <regex.Match object; span=(16, 18), match='på'>,
 <regex.Match object; span=(19, 27), match='Mårbacka'>,
 <regex.Match object; span=(28, 30), match='en'>,
 <regex.Match object; span=(31, 39), match='barnpiga'>,
 <regex.Match object; span=(41, 44), match='som'>,
 <regex.Match object; span=(45, 50), match='hette'>,
 <regex.Match object; span=(51, 55), match='Back'>,
 <regex.Match object; span=(56, 61), match='Kajsa'>]

#### Extracting indices

Write a `text_to_idx(words)` function to extract the indices from the list of tokens (words). Return a dictionary, where the keys will be the tokens (words), and the values a list of positions.

In [54]:
# Write your code here
def text_to_idx(words):
    """
    Builds an index from a list of match objects
    """
    word_idx = {}
    for word in words :
        match = word.captures()[0]
        # print("match =", match)
        pos = word.span()[0]
        # print("pos =", pos)
        word_idx[match] = word_idx.get(match,[]) + [pos]
    return word_idx

In [55]:
tokens = tokenize('En gång hade de på Mårbacka en barnpiga, som hette Back-Kajsa.'.lower().strip())
text_to_idx(tokens)

{'en': [0, 28],
 'gång': [3],
 'hade': [8],
 'de': [13],
 'på': [16],
 'mårbacka': [19],
 'barnpiga': [31],
 'som': [41],
 'hette': [45],
 'back': [51],
 'kajsa': [56]}

#### Reading one file

Read one file, _Mårbacka_, `marbacka.txt`, set it in lowercase, tokenize it, and index it. Call this index `idx`

In [60]:
# Write your code here
first_file = 'Selma/marbacka.txt'
with open(first_file, 'r', encoding='utf-8') as file :
    text = file.read().lower().strip()
idx = text_to_idx(tokenize(text))

In [61]:
idx['mårbacka']

[16,
 139,
 752,
 1700,
 2582,
 3324,
 15117,
 15404,
 27794,
 42175,
 49126,
 50407,
 52053,
 60144,
 63374,
 64910,
 67182,
 67330,
 67799,
 67824,
 69232,
 71328,
 72099,
 74147,
 74255,
 74614,
 76610,
 76884,
 77138,
 77509,
 77787,
 77936,
 78574,
 80597,
 81782,
 82003,
 84363,
 84786,
 85251,
 89837,
 97093,
 98642,
 100474,
 105063,
 105298,
 105721,
 108710,
 109133,
 112844,
 113725,
 114997,
 115583,
 115833,
 116368,
 116557,
 121896,
 124823,
 126409,
 126542,
 128758,
 130976,
 131939,
 132826,
 136914,
 137187,
 137872,
 139196,
 140721,
 142324,
 146781,
 151497,
 154335,
 155139,
 155438,
 155886,
 156405,
 158108,
 159817,
 160107,
 161158,
 162085,
 165847,
 168316,
 168528,
 169111,
 170333,
 172684,
 182047,
 182427,
 186362,
 189535,
 190999,
 191110,
 193177,
 196686,
 202552,
 206340,
 207789,
 208382,
 209874,
 210525,
 217464,
 219933,
 221393,
 221533,
 221880,
 222213,
 224190,
 229501,
 229598,
 230783,
 231453,
 232140,
 234427,
 236193,
 236950,
 240168,

#### Saving the index

Save your index in a file so that you can reuse it. Use a JSON file.

In [62]:
index_file = 'marbacka.json'
with open(index_file, 'w', encoding='utf-8') as fp:
    json.dump(idx, fp, ensure_ascii=False)

In [65]:
idx = {}
idx['mårbacka'] = []

Read back your file and store the content in `idx`

In [67]:
with open(index_file, 'r', encoding='utf-8') as f:
    idx = json.load(f)

In [68]:
idx['mårbacka']

[16,
 139,
 752,
 1700,
 2582,
 3324,
 15117,
 15404,
 27794,
 42175,
 49126,
 50407,
 52053,
 60144,
 63374,
 64910,
 67182,
 67330,
 67799,
 67824,
 69232,
 71328,
 72099,
 74147,
 74255,
 74614,
 76610,
 76884,
 77138,
 77509,
 77787,
 77936,
 78574,
 80597,
 81782,
 82003,
 84363,
 84786,
 85251,
 89837,
 97093,
 98642,
 100474,
 105063,
 105298,
 105721,
 108710,
 109133,
 112844,
 113725,
 114997,
 115583,
 115833,
 116368,
 116557,
 121896,
 124823,
 126409,
 126542,
 128758,
 130976,
 131939,
 132826,
 136914,
 137187,
 137872,
 139196,
 140721,
 142324,
 146781,
 151497,
 154335,
 155139,
 155438,
 155886,
 156405,
 158108,
 159817,
 160107,
 161158,
 162085,
 165847,
 168316,
 168528,
 169111,
 170333,
 172684,
 182047,
 182427,
 186362,
 189535,
 190999,
 191110,
 193177,
 196686,
 202552,
 206340,
 207789,
 208382,
 209874,
 210525,
 217464,
 219933,
 221393,
 221533,
 221880,
 222213,
 224190,
 229501,
 229598,
 230783,
 231453,
 232140,
 234427,
 236193,
 236950,
 240168,

### Reading the content of a folder

Write a `get_files(dir, suffix)` function that reads all the files in a folder with a specific `suffix` (txt). You will need the Python `os` package, see <a href="https://docs.python.org/3/library/os.html">https://docs.python.org/3/library/os.html</a>. You will return the file names in a list.

You can reuse this function:

In [69]:
def get_files(dir, suffix):
    """
    Returns all the files in a folder ending with suffix
    :param dir:
    :param suffix:
    :return: the list of file names
    """
    files = []
    for file in os.listdir(dir):
        if file.endswith(suffix):
            files.append(file)
    return files

In [71]:
# Write your code here
folder = 'Selma/'
corpus_files = get_files(folder, "txt")
corpus_files

['bannlyst.txt',
 'gosta.txt',
 'herrgard.txt',
 'jerusalem.txt',
 'kejsaren.txt',
 'marbacka.txt',
 'nils.txt',
 'osynliga.txt',
 'troll.txt']

### Creating a master index

Complete your program with the creation of master index, where you will associate each word of the corpus with the files, where it occur and its positions: a posting list.
Below is an except of the master index with the words <i>samlar</i> and <i>ände</i>:

In [16]:
{'samlar':
            {'troll.txt': [641880, 654233],
            'nils.txt': [51805, 118943],
            'osynliga.txt': [399121],
            'gosta.txt': [313784, 409998, 538165]},
 'ände':
            {'troll.txt': [39562, 650112],
            'kejsaren.txt': [50171],
            'marbacka.txt': [370324],
            'nils.txt': [1794],
            'osynliga.txt': [272144]}
}

{'samlar': {'troll.txt': [641880, 654233],
  'nils.txt': [51805, 118943],
  'osynliga.txt': [399121],
  'gosta.txt': [313784, 409998, 538165]},
 'ände': {'troll.txt': [39562, 650112],
  'kejsaren.txt': [50171],
  'marbacka.txt': [370324],
  'nils.txt': [1794],
  'osynliga.txt': [272144]}}

The word <i>samlar</i>, for instance, occurs three times in the gosta text at positions
            313784, 409998, and 538165.

In [72]:
# write your code here
master_index = {}
for file in corpus_files:
    text = open(folder + file, encoding='utf-8').read().lower().strip()
    words = tokenize(text)
    idx = text_to_idx(words)
    for word in idx:
        if word in master_index:
            master_index[word][file] = idx[word]
        else:
            master_index[word] = {}
            master_index[word][file] = idx[word]

In [73]:
master_index['samlar']

{'gosta.txt': [313784, 409998, 538165],
 'nils.txt': [51805, 118943],
 'osynliga.txt': [399121],
 'troll.txt': [641880, 654233]}

In [74]:
master_index['mårbacka']

{'marbacka.txt': [16,
  139,
  752,
  1700,
  2582,
  3324,
  15117,
  15404,
  27794,
  42175,
  49126,
  50407,
  52053,
  60144,
  63374,
  64910,
  67182,
  67330,
  67799,
  67824,
  69232,
  71328,
  72099,
  74147,
  74255,
  74614,
  76610,
  76884,
  77138,
  77509,
  77787,
  77936,
  78574,
  80597,
  81782,
  82003,
  84363,
  84786,
  85251,
  89837,
  97093,
  98642,
  100474,
  105063,
  105298,
  105721,
  108710,
  109133,
  112844,
  113725,
  114997,
  115583,
  115833,
  116368,
  116557,
  121896,
  124823,
  126409,
  126542,
  128758,
  130976,
  131939,
  132826,
  136914,
  137187,
  137872,
  139196,
  140721,
  142324,
  146781,
  151497,
  154335,
  155139,
  155438,
  155886,
  156405,
  158108,
  159817,
  160107,
  161158,
  162085,
  165847,
  168316,
  168528,
  169111,
  170333,
  172684,
  182047,
  182427,
  186362,
  189535,
  190999,
  191110,
  193177,
  196686,
  202552,
  206340,
  207789,
  208382,
  209874,
  210525,
  217464,
  219933,
  2213

Save your master index in a file and read it again

In [75]:
with open('master.json', 'w', encoding='utf-8') as fp:
    json.dump(master_index, fp, ensure_ascii=False)

In [76]:
with open('master.json', 'r', encoding='utf-8') as f:
    master_index = json.load(f)

In [116]:
master_index['samlar']

{'gosta.txt': [313784, 409998, 538165],
 'nils.txt': [51805, 118943],
 'osynliga.txt': [399121],
 'troll.txt': [641880, 654233]}

#### Concordances

Write a `concordance(word, master_index, window)` function to extract the concordances of a `word` within a window of `window` characters

In [104]:
# Write your code here
def concordance(word, master_index, window):
    dic = master_index[word]
    files_pos = dic.items()
    for file, pos in files_pos :
        text = open(folder + file, encoding='utf-8').read().lower().strip().replace("\n"," ")
        print(file)
        for place in pos :
            # print(place)
            start = place - window
            end = place + window
            # print(start)
            # print(end)
            print(" "*6," "*(-start) , text[max(start,0):min(end,len(text))])

In [107]:
concordance('samlar', master_index, 25)

gosta.txt
        om ligger nära borg, och samlar ihop ett litet mid
        lika förstämda.  men hon samlar upp allt detta som
        n ensam i livet.  därmed samlar han korten tillhop
nils.txt
         bara, att du i all hast samlar ihop så mycket bos
        ar stannat hemma, och nu samlar de sig för att int
osynliga.txt
         till höger i kärran och samlar just ihop tömmarna
troll.txt
        en örtkunnig läkare, som samlar in markens växter 
        älper dem, och medan hon samlar och handlar för de


### Representing Documents with tf-idf

Once you have created the index, you will represent each document in your corpus as a dictionary. The keys of these dictionaries will be the words and you will define the value of a word with the tf-idf metric: 
1. Read the description of the tf-idf measure on Wikipedia (<a href="https://en.wikipedia.org/wiki/Tf%E2%80%93idf">https://en.wikipedia.org/wiki/Tf-idf</a>)
2. After reading the description, you probably realized that there are multiple definitions of tf-idf. In this assignment, 
 * Tf will be the relative frequency of the term in the document and 
 * idf, the logarithm base 10 of the inverse document frequency.
        
You have below the tf-idf values for a few words. In our example, the word <i>gås</i> has the value 0 in bannlyst.txt and the value 0.000101001964 in nils.txt

<pre>
troll.txt
	känna	 0.0
	gås	 0.0
	nils	 2.148161748868631e-06
	et	 0.0
kejsaren.txt
	känna	 0.0
	gås	 0.0
	nils	 8.08284798629935e-06
	et	 8.273225429362848e-05
marbacka.txt
	känna	 0.0
	gås	 0.0
	nils	 7.582276564686669e-06
	et	 9.70107989686256e-06
herrgard.txt
	känna	 0.0
	gås	 0.0
	nils	 0.0
	et	 0.0
nils.txt
	känna	 0.0
	gås	 0.00010100196417506702
	nils	 0.00010164426900380124
	et	 0.0
osynliga.txt
	känna	 0.0
	gås	 0.0
	nils	 0.0
	et	 0.0
jerusalem.txt
	känna	 0.0
	gås	 0.0
	nils	 4.968292117670952e-06
	et	 0.0
bannlyst.txt
	känna	 0.0
	gås	 0.0
	nils	 0.0
	et	 0.0
gosta.txt
	känna	 0.0
	gås	 0.0
	nils	 0.0
	et	 0.0
</pre>

Conceptually, the tf-idf representation is a vector. In your program, you will keep this idea and use all the words in the corpus as keys: Each dictionary will include all the words of the corpus as keys. The value of the key is then possibly 0, meaning that the word in not in the document or is in all the documents as for the word `nils` in `gosta.tx`. 

As further work, you may think of optimizing this part.

In [159]:
corpus_files
len(master_index["selma"].keys())
math.log10(2)
len(list(tokenize("hej, jag hete 23 sdsd axel och jag gillar hej")))

9

In [170]:
# Write your code here
tfidf = {}

idf = {}
total_files = len(corpus_files)
for word in master_index.keys() :
    idf[word] = math.log10(total_files/len(master_index[word].keys()))

for file in corpus_files:
    tfidf[file] = {}
    text = open(folder + file, encoding='utf-8').read().lower().strip().replace('\n',' ')
    file_length = len(list(tokenize(text)))
    for word in master_index.keys() :
        word_count = len(master_index[word].get(file,[]))
        word_freq = word_count/file_length
        tfidf[file][word] = word_freq*idf[word]

In [172]:
print(master_index['nils'])
for file in corpus_files :
    print(file)
    print(len(master_index['nils'].get(file,[])))

{'jerusalem.txt': [171205, 324540, 324586], 'kejsaren.txt': [263335, 266581], 'marbacka.txt': [347421, 365256], 'nils.txt': [17, 3629, 18122, 18159, 49682, 61218, 99050, 99446, 100632, 100850, 107420, 122516, 138978, 159346, 187665, 204863, 205377, 215613, 281268, 285357, 285454, 292483, 292713, 292980, 294817, 303588, 308340, 317519, 333659, 361086, 405967, 406139, 406724, 414324, 459738, 503209, 563342, 581222, 602124, 626011, 636848, 667690, 667951, 674565, 682702, 695641, 701899, 702256, 704143, 707989, 759870, 779155, 866429, 866532, 867887, 868874, 874170, 938003, 992433, 992805, 1011668, 1019630, 1019962, 1020373, 1045606, 1063798, 1064583, 1065559, 1068924, 1079624, 1079782, 1080261, 1082400, 1083201, 1083350, 1085516, 1086605, 1086798, 1091380], 'troll.txt': [273705]}
bannlyst.txt
0
gosta.txt
0
herrgard.txt
0
jerusalem.txt
3
kejsaren.txt
2
marbacka.txt
2
nils.txt
79
osynliga.txt
0
troll.txt
1


In [173]:
tfidf['troll.txt']['känna']

0.0

In [174]:
tfidf['troll.txt']['nils']

2.127040446479182e-06

In [161]:
for file in corpus_files :
    print(file)
    # print(tfidf[file]['känna'])
    # print(tfidf[file]['gås'])
    print(tfidf[file]['nils'])
    # print(tfidf[file]['et'])
    print()

bannlyst.txt
0.0

gosta.txt
0.0

herrgard.txt
0.0

jerusalem.txt
4.8782846470039695e-06

kejsaren.txt
7.979011193176822e-06

marbacka.txt
7.5182972330777704e-06

nils.txt
0.00010041841355987142

osynliga.txt
0.0

troll.txt
2.127040446479182e-06



### Comparing Documents

Using the cosine similarity, compare all the pairs of documents with their tf-idf representation and present your results in a table. You will include this table in your report.

#### Cosine similarity

Write a function computing the cosine similarity between two documents: `cosine_similarity(document1, document2)`

In [181]:
np.array(list(tfidf['nils.txt'].values()))

array([0.00000000e+00, 2.54711926e-07, 0.00000000e+00, ...,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00], shape=(41852,))

In [182]:
# Write your code here
def cosine_similarity(document1, document2):
    u = np.array(list(tfidf[document1].values()))
    v = np.array(list(tfidf[document2].values()))
    u_norm = np.linalg.norm(u)
    v_norm = np.linalg.norm(v)
    u_dot_v = u @ v
    return u_dot_v / (u_norm * v_norm)

In [183]:
cosine_similarity('herrgard.txt', 'jerusalem.txt')

np.float64(0.3463077343036067)

In [184]:
cosine_similarity('nils.txt', 'nils.txt')

np.float64(0.9999999999999999)

#### Similarity matrix

Compute the similarity matrix between the documents of the corpus. While computing the similarities, you will record the two most similar documents that you will call `most_sim_doc1` and `most_sim_doc2`.

In [201]:
# Write your code here
max_similarity = 0.0
most_sim_doc1 = ''
most_sim_doc2 = ''
output1 = " "*16
output2 = {}
for i, doc1 in enumerate(corpus_files) :
    output1 += f'{doc1[:-4]: <11}'
    output2[i] = f"{doc1: <16}"
    for doc2 in corpus_files :
        cos = cosine_similarity(doc1, doc2)
        if cos > max_similarity and doc1 != doc2:
            max_similarity = cos
            most_sim_doc1 = doc1
            most_sim_doc2 = doc2
        output2[i] += f'{cos:.4f}     '

print(output1)
for output in output2.values() :
    print(output)

                bannlyst   gosta      herrgard   jerusalem  kejsaren   marbacka   nils       osynliga   troll      
bannlyst.txt    1.0000     0.0439     0.0009     0.1661     0.2075     0.1941     0.2987     0.0468     0.0559     
gosta.txt       0.0439     1.0000     0.0031     0.0041     0.0436     0.0746     0.0859     0.1253     0.0535     
herrgard.txt    0.0009     0.0031     1.0000     0.3463     0.0007     0.0034     0.0043     0.0048     0.0011     
jerusalem.txt   0.1661     0.0041     0.3463     1.0000     0.1487     0.1319     0.2077     0.0264     0.0284     
kejsaren.txt    0.2075     0.0436     0.0007     0.1487     1.0000     0.2092     0.2764     0.0466     0.0761     
marbacka.txt    0.1941     0.0746     0.0034     0.1319     0.2092     1.0000     0.2730     0.0869     0.0649     
nils.txt        0.2987     0.0859     0.0043     0.2077     0.2764     0.2730     1.0000     0.0909     0.0859     
osynliga.txt    0.0468     0.1253     0.0048     0.0264     0.0466     0

Give the name of the two novels that are the most similar.

In [194]:
print("Most similar:", most_sim_doc1, most_sim_doc2, "Similarity:", max_similarity)

Most similar: herrgard.txt jerusalem.txt Similarity: 0.3463077343036067


<h2>Turning in your assigment</h2>

Now your are done with the program. To complete this assignment, you will:
1. Showcase the concordances of your program with a Hugging Face Gradio app,
2. Write a short individual report on your program, 
3. Read the text <i>Challenges in Building Large-Scale Information Retrieval Systems</i> about the history of <a href="https://research.google.com/people/jeff/WSDM09-keynote.pdf">Google indexing</a> by <a href="https://research.google.com/pubs/jeff.html">Jeff Dean</a> and write a short comment on it. See the details below.
4. Watch the speech delivered by Steve Jobs in 1985 (https://www.youtube.com/watch?v=NT6oUsn3ANA) and describe superficially in one paragraph how computers can amass knowledge and have dialogues with us.

For the Gradio program, you will:
1. Create a Hugging Face account,
2. Upload your master index and Selma's novels as a dataset. It is all drag-and-drop,
3. Create the app as a space where you will extract the concordances of a word. You will reuse your `concordance` function. The creation process is also interactive and you just have to edit the app.py and requirements.txt files.

As it involves Hugging Face functions that are not part of the course, you can use a large language model to help you. You can also look at this app and its files and reuse the structure: https://huggingface.co/spaces/pnugues/selma_sim

You will submit your report as well as your notebook (for archiving purposes) to Canvas: <https://canvas.education.lu.se/>. You have two separate places: one of the report and the other for the program.

I suggest that you use this structure for your report:
1. Objectives and dataset
2. Method and program structure, where you should outline your program and possibly describe difficult parts. 
3. Results.
4. Conclusion
5. Answer to possible questions

In your report of about two pages:
1. You will describe your indexer (Sect. Method) and comment your results (Sect. Result); in this description, you will write the regular expression you used for the tokenization (Sect. Method) and include the similarity matrix and the URL of your Hugging Face demo (Sect. Results);
2. In Jeff Dean's document, slide 45 describes an indexing technique. Tell how your index encoding is related to what Google did. (Sect. Other questions).
3. Referring to Jobs's speech, comment shortly how computers can amass knowledge and have dialogues with us.

To write your report, use Latex. This is mandatory as it will help you structure your text. You will then upload a PDF file in Canvas. You can try Overleaf (<a href="https://www.overleaf.com/">www.overleaf.com</a>) as it makes it easy to create Latex document.

The submission deadline is September 18, 2026. You will have only three submission attempts. The deadline for the second and third ones are one week after you are noticed of your result.